# Linear Regression - Example
In this notebook we demonstrate a Machine Learning workflow using a train, validation and test set. 

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.datasets import load_diabetes

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error

ModuleNotFoundError: No module named 'numpy'

# Loading Data

In [ ]:
# This code is merely executed to make it easy to inspect the dataset description
diabetes = load_diabetes()

In [ ]:
# print(diabetes.DESCR)

## Storing/Loading the data in the way it will be used

In [ ]:
X, y = load_diabetes(return_X_y=True, as_frame=True)

# Only choose two variables for our models
X = X[['bmi', 'bp']]

# Give the target a descriptive name
y.name = 'disease_progression'

In [ ]:
print(X.info())

In [ ]:
print(y.info())

# Train, Validation, and Test Set

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=40)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.3, random_state=36)

# EDA
We are only allowed to explore and learn things from the training data when creating our model.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4), layout='constrained')

axs[0].scatter(X_train['bmi'], y_train)
axs[0].set_xlabel('BMI (standardized)')
axs[0].set_ylabel('Disease progression')
axs[0].set_title('BMI vs. target (Train Data)')

axs[1].scatter(X_train['bp'], y_train, alpha=0.7, color='red')
axs[1].set_xlabel('Blood pressure (standardized)')
axs[1].set_ylabel('Disease progression')
axs[1].set_title('Blood pressure vs. target (Train Data)')

In [ ]:
X_train.head()

In [ ]:
y_train.head()

In [ ]:
y_train.describe()

# Training 2 different models

In [ ]:
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
lin_reg_pred = lin_reg.predict(X_val)

In [ ]:
tree_reg = DecisionTreeRegressor(random_state=42)

# Find the best hyperparameter through GridSearch.
hyper_params = {'max_depth': (None, 1, 2, 3, 5, 10)}
reg = GridSearchCV(
    tree_reg,
    hyper_params,
    scoring='neg_root_mean_squared_error',
    cv=5
)

reg.fit(X_train, y_train)
tree_reg_pred = reg.predict(X_val)

In [ ]:
print(reg.best_params_)
pd.DataFrame(reg.cv_results_)

## Choosing the best model through validation set
For MAE and RMSE, lower is better. We use RMSE as the primary metric for choosing a model.

In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred))
    }

validation_results = pd.DataFrame({
    'Linear Regression': regression_metrics(y_val, lin_reg_pred),
    'Decision Tree': regression_metrics(y_val, tree_reg_pred)
}).T

validation_results

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4), layout='constrained')

# Linear regression
lin_reg_residuals = y_val - lin_reg_pred

axs[0].scatter(lin_reg_pred, lin_reg_residuals, alpha=0.7)
axs[0].axhline(0, color='black', linestyle='--')
axs[0].set_xlabel('Predicted value')
axs[0].set_ylabel('Residual (actual - predicted)')
axs[0].set_title('Linear Regression')

# Decision tree
tree_reg_residuals = y_val - tree_reg_pred

axs[1].scatter(tree_reg_pred, tree_reg_residuals, alpha=0.7)
axs[1].axhline(0, color='black', linestyle='--')
axs[1].set_xlabel('Predicted value')
axs[1].set_ylabel('Residual (actual - predicted)')
axs[1].set_title('Decision Tree')

# Evaluating chosen model through test set
The linear regression model has the lower validation RMSE for this split, so we choose it. We now retrain it using the combined training and validation data, then evaluate it once on the untouched test set. 

**Question: What is the purpose with test data?**

In [ ]:
# Now we retrain our chosen model on the train + validation data.
lin_reg_final = LinearRegression().fit(X_train_full, y_train_full)
pred_test = lin_reg_final.predict(X_test)

In [ ]:
test_results = pd.Series(
    regression_metrics(y_test, pred_test),
    name='Test set'
)
test_results